# Task 0: Dataset Construction
This task builds the training corpus of 1000 Indian names, cleans each name, and stores the final list in TrainingNames.txt. The output is used as the character-level supervision set for all sequence models.

In [ ]:
# Core libraries for modeling, training, and utilities
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import random
import math
import string

# Reproducibility setup
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
# Task-0 dataset builder: create a clean list of 1000 Indian first names for character-level modeling.
# If the package is not installed yet, uncomment and run the next line once.
# %pip install -q indian-names

import re 
import random  
import indian_names  

random.seed(42) 

def clean_name(s):
    # Convert any input to string, remove non-alphabetic characters, and trim spaces.
    s = re.sub(r"[^A-Za-z]", "", str(s)).strip()
    # Filter out very short or very long forms to avoid noisy/unnatural training samples.
    if len(s) < 3 or len(s) > 15:
        return ""
    # Use lowercase for a consistent character vocabulary later in preprocessing.
    return s.lower()

names_set = set()  # Set ensures all collected names remain unique.
max_tries = 20000  # Safety cap so loop does not run forever in edge cases.
tries = 0  # Counter to track how many draw attempts were made.

# Collect names until we have 1000 unique clean entries or hit safety limit.
while len(names_set) < 1000 and tries < max_tries:
    tries += 1  # Increment attempt counter each iteration.
    # Randomly choose gender for broader style coverage in generated names.
    g = "male" if random.random() < 0.5 else "female"
    # Pull one name from library and clean it before adding.
    nm = clean_name(indian_names.get_first_name(gender=g))
    if nm:  # Add only non-empty cleaned outputs.
        names_set.add(nm)

# Fail fast if collection did not reach target size, so user can re-run/debug.
if len(names_set) < 1000:
    raise ValueError(f"Could collect only {len(names_set)} unique names. Re-run cell.")

# Convert set to sorted list for deterministic file order and easier diff/review.
names = sorted(list(names_set))[:1000]

# Save one name per line; this file is used as input for later modeling tasks.
with open("TrainingNames.txt", "w", encoding="utf-8") as f:
    for n in names:
        f.write(n + "\n")

# Print summary + quick sample for manual sanity check.
print(f"Saved {len(names)} real Indian names to TrainingNames.txt")
print("Sample:", names[:20])

Saved 1000 real Indian names to TrainingNames.txt
Sample: ['aanchal', 'aaradhita', 'aarohi', 'aarti', 'aastha', 'abha', 'abhay', 'abhigyan', 'abhijeet', 'abhilash', 'abhilasha', 'abhilesh', 'abhinav', 'abhisar', 'abhishek', 'adhya', 'aditi', 'aditya', 'adityanath', 'adya']


In [ ]:
# Define special boundary/padding symbols used in character-level sequence modeling.
SOS_TOKEN = '<'  
EOS_TOKEN = '>' 
PAD_TOKEN = '#'  

# Load the cleaned Task-0 name list generated in the previous step.
with open("TrainingNames.txt", 'r', encoding='utf-8') as f:
    # Normalize to lowercase so character vocabulary is consistent and compact.
    names = [line.strip().lower() for line in f.readlines()]

# Build a character vocabulary from all names in the dataset.
chars = set(''.join(names)) 
chars.add(SOS_TOKEN) 
chars.add(EOS_TOKEN)
chars.add(PAD_TOKEN)  

# Create lookup maps for encoding and decoding.
char_to_idx = {char: idx for idx, char in enumerate(sorted(list(chars)))}  # Character -> integer id.
idx_to_char = {idx: char for char, idx in char_to_idx.items()}  # Integer id -> character.
vocab_size = len(char_to_idx)  # Total number of symbols used by the model.

# Shared hyperparameters used for all Task-1 model variants (for fair comparison).
EMBEDDING_DIM = 48  
NUM_LAYERS = 2
LEARNING_RATE = 0.002  
EPOCHS = 50  
BATCH_SIZE = 64  

def name_to_tensor(name):
    # Encode one name as indices in the format: <SOS> + characters + <EOS>.
    indices = [char_to_idx[SOS_TOKEN]] + [char_to_idx[c] for c in name] + [char_to_idx[EOS_TOKEN]]
    return torch.tensor(indices, dtype=torch.long)  # Return 1D tensor of token ids.

# Compute max sequence length after adding SOS/EOS so every sample can be padded uniformly.
max_len = max([len(n) for n in names]) + 2

def pad_sequence(tensor, max_length):
    # Determine how many PAD tokens are required to reach fixed max length.
    pad_len = max_length - len(tensor)
    # Pad on the right with PAD token id so model input shape is consistent.
    return F.pad(tensor, (0, pad_len), value=char_to_idx[PAD_TOKEN])

# Convert every name into a padded tensor and stack into one dataset tensor.
dataset = [pad_sequence(name_to_tensor(n), max_len) for n in names]  # List of [max_len] tensors.
dataset = torch.stack(dataset).to(device)  # Final shape: [num_names, max_len], moved to active device.

# Print quick sanity stats for verification before model training.
print(f"Dataset Size: {dataset.shape}")
print(f"Vocabulary Size: {vocab_size}")

Dataset Size: torch.Size([1000, 16])
Vocabulary Size: 27


In [ ]:
# Inspect one encoded training sequence to verify preprocessing correctness.
print("Sample tensor:", dataset[1])  # Shows SOS/name/EOS/PAD pattern in index form.

print(f"SOS token index: {char_to_idx[SOS_TOKEN]}")  # Start-of-sequence token id.
print(f"EOS token index: {char_to_idx[EOS_TOKEN]}")  # End-of-sequence token id.
print(f"PAD token index: {char_to_idx[PAD_TOKEN]}")  # Padding token id used in batching.

# These quick prints are useful before model training to catch token-map mistakes early.

Sample tensor: tensor([ 1,  3,  3, 19,  3,  6, 10, 11, 21,  3,  2,  0,  0,  0,  0,  0])
SOS token index: 1
EOS token index: 2
PAD token index: 0


# Task 1: Model Implementation (From Scratch)
In this task, three recurrent architectures are implemented with custom model classes and explicit training loops: Vanilla RNN, BLSTM, and RNN with additive attention. For each model, architecture, parameter count, and hyperparameters are reported.

### Model 1: Vanilla RNN
A simple autoregressive baseline with character embeddings, recurrent hidden state updates, and linear token prediction at each time step.

In [ ]:
class VanillaCharRNN(nn.Module):
    # Vanilla character-level recurrent baseline used in Task-1 comparison.
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=1):
        super(VanillaCharRNN, self).__init__()  # Initialize nn.Module internals.
        self.hidden_dim = hidden_dim  # Store hidden size for reference/debugging.
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=char_to_idx[PAD_TOKEN])  # Character lookup table.
        self.rnn = nn.RNN(embedding_dim, hidden_dim, num_layers, batch_first=True)  # Core recurrent transition.
        self.fc = nn.Linear(hidden_dim, vocab_size)  # Map hidden state to next-character logits.

    def forward(self, x, hidden=None):
        # x is [batch, seq_len] of character indices.
        embedded = self.embedding(x)  # Convert indices -> dense vectors.
        out, hidden = self.rnn(embedded, hidden)  # Recurrently process sequence.
        logits = self.fc(out)  # Produce token logits at every time step.
        return logits, hidden  # Return both logits and final hidden state.

# Instantiate model with assignment hyperparameters.
model_rnn = VanillaCharRNN(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS).to(device)
params_rnn = sum(p.numel() for p in model_rnn.parameters() if p.requires_grad)  # Count trainable parameters.
print(f"Vanilla RNN - Trainable Parameters: {params_rnn}")  # Required reporting output.
print("Architecture: Embedding -> RNN -> Linear(vocab)")  # Required architecture description.
print(f"Hyperparameters: hidden_dim={HIDDEN_DIM}, num_layers={NUM_LAYERS}, lr={LEARNING_RATE}")  # Required hyperparameter description.

def train_autoregressive(model, data, epochs=EPOCHS):
    # Shared training loop for autoregressive models (Vanilla RNN and BLSTM).
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)  # Adam optimizer for stable training.
    criterion = nn.CrossEntropyLoss(ignore_index=char_to_idx[PAD_TOKEN])  # Ignore PAD positions in loss.

    model.train()  # Enable training mode.
    for epoch in range(epochs):
        optimizer.zero_grad()  # Reset gradients from previous update.

        # Teacher-forcing setup: input excludes last token, target excludes first token.
        inputs = data[:, :-1]  # [batch, seq_len-1]
        targets = data[:, 1:]  # Next-character labels.

        logits, _ = model(inputs)  # Forward pass through model.
        loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1))  # Flatten time dimension for CE.

        loss.backward()  # Backpropagate gradients.
        optimizer.step()  # Update parameters once per epoch (full-batch style).

        if (epoch + 1) % 10 == 0:  # Print periodic progress for report logging.
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

print("Training Vanilla RNN...")  # Training start message.
train_autoregressive(model_rnn, dataset)  # Execute training for model-1.

Vanilla RNN - Trainable Parameters: 60587
Architecture: Embedding -> RNN -> Linear(vocab)
Hyperparameters: hidden_dim=128, num_layers=2, lr=0.002
Training Vanilla RNN...
Epoch 10/50, Loss: 2.4002
Epoch 20/50, Loss: 2.1518
Epoch 30/50, Loss: 2.0284
Epoch 40/50, Loss: 1.9460
Epoch 50/50, Loss: 1.8799


In [ ]:
# Report 1: trainable parameter count (required by assignment Task-1).
params_rnn = sum(p.numel() for p in model_rnn.parameters() if p.requires_grad)  # Count learnable weights only.
print(f"Vanilla RNN trainable parameters: {params_rnn:,}")

# Report 2: theoretical memory footprint based on dtype and parameter count.
bytes_per_param = next(model_rnn.parameters()).element_size()  # Usually 4 bytes for float32.
size_mb_theoretical = (params_rnn * bytes_per_param) / (1024 ** 2)  # Convert bytes to megabytes.
print(f"Theoretical model size: {size_mb_theoretical:.4f} MB")

# Report 3: actual serialized state_dict size saved on disk.
import os  # Used here only for file-size inspection.
save_path = "vanilla_rnn_state_dict.pt"  # Output checkpoint path.
torch.save(model_rnn.state_dict(), save_path)  # Save weights for reproducibility/submission.
size_mb_saved = os.path.getsize(save_path) / (1024 ** 2)  # Measure file size in MB.
print(f"Saved state_dict size: {size_mb_saved:.4f} MB at {save_path}")

Vanilla RNN trainable parameters: 60,587
Theoretical model size: 0.2311 MB
Saved state_dict size: 0.2355 MB at vanilla_rnn_state_dict.pt


### Model 2: Bidirectional LSTM (BLSTM)
A stronger recurrent baseline that captures both left and right context before projecting to vocabulary logits.

In [ ]:
class BLSTMCharModel(nn.Module):
    # Bidirectional LSTM baseline: uses past+future context at each time step.
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=1):
        super(BLSTMCharModel, self).__init__()  # Initialize nn.Module internals.
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=char_to_idx[PAD_TOKEN])  # Character embeddings.
        self.blstm = nn.LSTM(
            embedding_dim,  # Input size to recurrent layer.
            hidden_dim,  # Hidden state size per direction.
            num_layers=num_layers,  # Number of stacked BLSTM layers.
            batch_first=True,  # Input layout: [batch, time, feature].
            bidirectional=True  # Enable forward and backward recurrence.
        )
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)  # Concatenate both directions before projection.

    def forward(self, x, hidden=None):
        embedded = self.embedding(x)  # Convert token ids to vectors.
        out, hidden = self.blstm(embedded, hidden)  # Process sequence bidirectionally.
        logits = self.fc(out)  # Map recurrent outputs to vocabulary logits.
        return logits, hidden  # Return logits and final hidden/cell states.

# Instantiate BLSTM and print assignment-required model details.
model_blstm = BLSTMCharModel(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS).to(device)
params_blstm = sum(p.numel() for p in model_blstm.parameters() if p.requires_grad)  # Trainable-parameter count.
print(f"BLSTM - Trainable Parameters: {params_blstm}")
print("Architecture: Embedding -> Bidirectional LSTM -> Linear(vocab)")
print(f"Hyperparameters: hidden_dim={HIDDEN_DIM}, num_layers={NUM_LAYERS}, lr={LEARNING_RATE}")

print("Training BLSTM...")  # Start BLSTM training using shared autoregressive trainer.
train_autoregressive(model_blstm, dataset)

BLSTM - Trainable Parameters: 585771
Architecture: Embedding -> Bidirectional LSTM -> Linear(vocab)
Hyperparameters: hidden_dim=128, num_layers=2, lr=0.002
Training BLSTM...
Epoch 10/50, Loss: 2.4911
Epoch 20/50, Loss: 1.8634
Epoch 30/50, Loss: 1.0990
Epoch 40/50, Loss: 0.4430
Epoch 50/50, Loss: 0.1447


### Model 3: RNN with Basic Additive Attention
An encoder-decoder style model where additive attention dynamically selects useful encoder states at each decoding step.

In [ ]:
class Seq2SeqAttention(nn.Module):
    # Attention-based encoder-decoder for character generation.
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(Seq2SeqAttention, self).__init__()  # Initialize parent nn.Module.
        self.hidden_dim = hidden_dim  # Store hidden size for layer construction.
        self.vocab_size = vocab_size  # Store vocabulary size for output projection.
        self.pad_idx = char_to_idx[PAD_TOKEN]  # PAD index used for masking attention.

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=self.pad_idx)  # Shared embedding lookup.
        self.encoder = nn.RNN(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)  # BiRNN encoder.

        # Additive attention: score = v^T tanh(W [decoder_hidden ; encoder_output]).
        self.attn = nn.Linear(hidden_dim * 3, hidden_dim)  # W projection in additive attention.
        self.v = nn.Linear(hidden_dim, 1, bias=False)  # v vector projection to scalar score.

        self.decoder = nn.RNN(embedding_dim + hidden_dim * 2, hidden_dim, batch_first=True)  # Decoder consumes token+context.
        self.fc = nn.Linear(hidden_dim, vocab_size)  # Final logits over vocabulary.

    def forward(self, src, trg):
        batch_size = src.size(0)  # Number of sequences in batch.
        src_len = src.size(1)  # Encoder sequence length.
        trg_len = trg.size(1)  # Decoder input length.

        src_mask = (src != self.pad_idx)  # Mask real tokens to suppress PAD attention.

        embedded_src = self.embedding(src)  # Embed source sequence.
        encoder_outputs, hidden = self.encoder(embedded_src)  # Run bidirectional encoder.

        hidden = torch.tanh(hidden[0] + hidden[1]).unsqueeze(0)  # Merge directions for decoder init.

        embedded_trg = self.embedding(trg)  # Embed decoder inputs (teacher forcing).
        outputs = torch.zeros(batch_size, trg_len, self.vocab_size, device=src.device)  # Preallocate logits tensor.

        for t in range(trg_len):  # Decode one step at a time.
            hidden_expanded = hidden.transpose(0, 1).repeat(1, src_len, 1)  # Broadcast decoder state across source length.
            energy_input = torch.cat((hidden_expanded, encoder_outputs), dim=2)  # Concatenate for attention scoring.
            energy = torch.tanh(self.attn(energy_input))  # Nonlinear attention features.
            attn_scores = self.v(energy).squeeze(2)  # Scalar score per source position.

            attn_scores = attn_scores.masked_fill(~src_mask, -1e9)  # Force PAD positions to near-zero probability.
            attention = F.softmax(attn_scores, dim=1).unsqueeze(1)  # Normalize to attention weights.

            context = torch.bmm(attention, encoder_outputs)  # Weighted sum of encoder states.

            dec_input = embedded_trg[:, t, :].unsqueeze(1)  # Current decoder token embedding.
            rnn_input = torch.cat((dec_input, context), dim=2)  # Concatenate token and attention context.

            out, hidden = self.decoder(rnn_input, hidden)  # Decoder transition.
            outputs[:, t, :] = self.fc(out.squeeze(1))  # Save logits for this step.

        return outputs  # [batch, trg_len, vocab_size]

def train_seq2seq(model, data, epochs=EPOCHS):
    # Dedicated trainer for attention model (separate from autoregressive RNN/BLSTM trainer).
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)  # Adam optimizer.
    criterion = nn.CrossEntropyLoss(ignore_index=char_to_idx[PAD_TOKEN])  # Ignore PAD targets in loss.

    model.train()  # Enable training behavior.
    for epoch in range(epochs):
        optimizer.zero_grad(set_to_none=True)  # Clear gradients efficiently.

        src = data  # Source sequence for encoder.
        trg_input = data[:, :-1]  # Decoder input prefix.
        trg_target = data[:, 1:]  # Next-token ground truth.

        logits = model(src, trg_input)  # Forward pass through seq2seq attention model.
        loss = criterion(logits.reshape(-1, vocab_size), trg_target.reshape(-1))  # Flatten sequence dimension for CE.

        loss.backward()  # Backpropagate loss.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Stabilize training with gradient clipping.
        optimizer.step()  # Update parameters.

        if (epoch + 1) % 10 == 0:  # Periodic training log for report.
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

model_attn = Seq2SeqAttention(vocab_size, EMBEDDING_DIM, HIDDEN_DIM).to(device)  # Instantiate attention model.
params_attn = sum(p.numel() for p in model_attn.parameters() if p.requires_grad)  # Trainable parameter count.
print(f"RNN with Attention - Trainable Parameters: {params_attn}")
print("Architecture: RNN encoder -> additive attention -> RNN decoder -> Linear(vocab)")
print(f"Hyperparameters: hidden_dim={HIDDEN_DIM}, num_layers=1, lr={LEARNING_RATE}")

print("Training Seq2Seq with Attention...")
train_seq2seq(model_attn, dataset)

RNN with Attention - Trainable Parameters: 155307
Architecture: RNN encoder -> additive attention -> RNN decoder -> Linear(vocab)
Hyperparameters: hidden_dim=128, num_layers=1, lr=0.002
Training Seq2Seq with Attention...
Epoch 10/50, Loss: 2.3530
Epoch 20/50, Loss: 1.8475
Epoch 30/50, Loss: 1.3508
Epoch 40/50, Loss: 0.9496
Epoch 50/50, Loss: 0.4645


# Task 2: Quantitative Evaluation
This task evaluates generated names using novelty rate and diversity so the three architectures can be compared with the same generation budget.

In [ ]:
def generate_from_autoregressive(model, num_names=200):
    # Generate names token-by-token using an autoregressive recurrent model.
    model.eval()  # Switch to inference mode.
    generated = []  # Store final generated names.
    with torch.no_grad():  # Disable gradient tracking for faster inference.
        for _ in range(num_names):  # Produce requested number of names.
            input_seq = torch.tensor([[char_to_idx[SOS_TOKEN]]]).to(device)  # Start from SOS token.
            hidden = None  # Let model initialize hidden state internally.
            name = ""  # Collect generated characters here.
            for _ in range(15):  # Cap maximum generated length.
                logits, hidden = model(input_seq, hidden)  # Predict next-token distribution.
                probs = F.softmax(logits[:, -1, :] / 0.8, dim=-1)  # Temperature sampling for diversity.
                next_char_idx = torch.multinomial(probs, 1).item()  # Sample one token index.

                if next_char_idx == char_to_idx[EOS_TOKEN]:  # Stop when EOS is sampled.
                    break
                if next_char_idx != char_to_idx[PAD_TOKEN]:  # Ignore PAD in generated text.
                    name += idx_to_char[next_char_idx]  # Append decoded character.
                input_seq = torch.tensor([[next_char_idx]]).to(device)  # Feed sampled token back as next input.
            if len(name) > 2:  # Keep only reasonably formed outputs.
                generated.append(name.capitalize())  # Store title-cased name.
    return generated  # Return generated list.

def generate_from_seq2seq(model, num_names=200):
    # Generate names using attention model with source-conditioned decoding.
    model.eval()  # Switch to inference mode.
    generated = []  # Store generated names.
    with torch.no_grad():  # No gradients needed at evaluation time.
        for _ in range(num_names):
            rand_idx = random.randint(0, len(dataset) - 1)  # Pick one source sequence from dataset.
            src = dataset[rand_idx].unsqueeze(0)  # Shape [1, src_len].

            trg = torch.tensor([[char_to_idx[SOS_TOKEN]]]).to(device)  # Decoder always starts with SOS.
            name = ""  # Accumulate predicted characters.
            for _ in range(15):  # Maximum decode length.
                logits = model(src, trg)  # Run decoder on current target prefix.
                probs = F.softmax(logits[:, -1, :] / 0.8, dim=-1)  # Temperature-scaled distribution.
                next_char_idx = torch.multinomial(probs, 1).item()  # Sample next character.

                if next_char_idx == char_to_idx[EOS_TOKEN]:  # Stop on EOS.
                    break
                if next_char_idx != char_to_idx[PAD_TOKEN]:  # Ignore PAD artifacts.
                    name += idx_to_char[next_char_idx]
                trg = torch.cat((trg, torch.tensor([[next_char_idx]]).to(device)), dim=1)  # Extend decoder input prefix.

            if len(name) > 2:
                generated.append(name.capitalize())
    return generated

# Generate equal-sized sample sets for fair quantitative comparison.
print("Generating evaluation samples...")
gen_rnn = generate_from_autoregressive(model_rnn, 500)  # 500 names from Vanilla RNN.
gen_blstm = generate_from_autoregressive(model_blstm, 500)  # 500 names from BLSTM.
gen_attn = generate_from_seq2seq(model_attn, 500)  # 500 names from attention model.

train_set = set([n.lower() for n in names])  # Reference set for novelty computation.

def evaluate(generated_list, model_name):
    # Compute assignment metrics: novelty rate and diversity.
    if len(generated_list) == 0:  # Guard for pathological generation failures.
        return
    novel = [n for n in generated_list if n.lower() not in train_set]  # Names not present in training data.
    novelty_rate = len(novel) / len(generated_list)  # Fraction of unseen names.

    unique_names = set(generated_list)  # Distinct generated strings.
    diversity = len(unique_names) / len(generated_list)  # Unique-to-total ratio.

    print(f"--- {model_name} ---")  # Header for model-specific result block.
    print(f"Novelty Rate: {novelty_rate * 100:.2f}%")  # Percentage format for report.
    print(f"Diversity:    {diversity * 100:.2f}%\n")

evaluate(gen_rnn, "Vanilla RNN")  # Evaluate model-1.
evaluate(gen_blstm, "BLSTM")  # Evaluate model-2.
evaluate(gen_attn, "RNN with Attention")  # Evaluate model-3.

Generating evaluation samples...
--- Vanilla RNN ---
Novelty Rate: 97.19%
Diversity:    99.40%

--- BLSTM ---
Novelty Rate: 100.00%
Diversity:    73.79%

--- RNN with Attention ---
Novelty Rate: 75.75%
Diversity:    95.39%



# Task 3: Qualitative Analysis
This section prints representative names from each model to inspect realism, repetition artifacts, and stylistic differences that are not captured by metrics alone.

In [ ]:
# Print representative qualitative samples from each model output set.
print("Representative Generated Samples:\n")

# Sample up to 10 names from Vanilla RNN generations.
print(f"Vanilla RNN: \n{random.sample(gen_rnn, min(10, len(gen_rnn)))}\n")

# Sample up to 10 names from BLSTM generations.
print(f"BLSTM: \n{random.sample(gen_blstm, min(10, len(gen_blstm)))}\n")

# Sample up to 10 names from attention-model generations.
print(f"RNN w/ Attention: \n{random.sample(gen_attn, min(10, len(gen_attn)))}\n")

Representative Generated Samples:

Vanilla RNN: 
['Ansa', 'Rashithar', 'Prayat', 'Nari', 'Dildap', 'Saneravi', 'Parshita', 'Areli', 'Hurina', 'Saminda']

BLSTM: 
['Uuuuuuuuuuuuuuu', 'Jvegeeeeeeeeeee', 'Ggggggggggggggg', 'Uauauauyuyyyyyy', 'Kakakakakakakak', 'Uuuuuuuuuuuuuuu', 'Gjgjjjjgjjjjjjo', 'Nknkkkkkkkkkkkk', 'Kakakkkkkkkkkyk', 'Padadadadadadad']

RNN w/ Attention: 
['Rushita', 'Sharanaji', 'Mona', 'Rathana', 'Drantadyi', 'Prajjslal', 'Bharav', 'Vilac', 'Rajkudra', 'Ishika']

